In [ ]:
# In[1]:

import pandas as pd

# load CSVs
metric_df = pd.read_csv("metric.csv")
trace_df = pd.read_csv("trace.csv")
log_df = pd.read_csv("log.csv")
error_df = pd.read_csv("error_logs.csv")

# parse timestamps to UTC datetimes per rules
for df in (metric_df, trace_df, log_df, error_df):
    df['timestamp'] = pd.to_datetime(df['timestamp'], unit='s', utc=True)

# incident window (UTC)
window_start = pd.Timestamp("2024-02-21 04:04:01", tz="UTC")
window_end   = pd.Timestamp("2024-02-21 04:34:01", tz="UTC")

# helper to aggregate metric-like files
def aggregate_timeseries(df, source, name_col):
    # ensure value numeric when present
    if 'value' in df.columns:
        df['value'] = pd.to_numeric(df['value'], errors='coerce')
    grp = df.groupby(['cmdb_id', name_col], dropna=False)
    # total points
    total_points = grp.size().rename('total_points')
    # global p95 computed on the entire series per group (do not pre-filter)
    if 'value' in df.columns:
        global_p95 = grp['value'].quantile(0.95).rename('global_p95')
    else:
        global_p95 = pd.Series(index=total_points.index, dtype='float64', name='global_p95')
    earliest = grp['timestamp'].min().rename('earliest_ts')
    latest = grp['timestamp'].max().rename('latest_ts')
    # points in window: count rows whose timestamp within the window
    in_window_mask = df['timestamp'].between(window_start, window_end)
    points_in_window = df.loc[in_window_mask].groupby(['cmdb_id', name_col]).size().reindex(total_points.index, fill_value=0).rename('points_in_window')
    # combine
    out = pd.concat([total_points, global_p95, points_in_window, earliest, latest], axis=1).reset_index()
    out = out.rename(columns={name_col: 'name'})
    out.insert(0, 'source', source)
    # sort by total_points desc then source,name
    out = out.sort_values(['total_points', 'source', 'cmdb_id', 'name'], ascending=[False, True, True, True]).reset_index(drop=True)
    return out

# aggregate metric, trace, log
metric_agg = aggregate_timeseries(metric_df, source='metric', name_col='kpi_name')
trace_agg  = aggregate_timeseries(trace_df,  source='trace',  name_col='trace_name')
log_agg    = aggregate_timeseries(log_df,    source='log',    name_col='log_name')

# limit rows to top 50 per requirement
metric_agg_top = metric_agg.head(50)
trace_agg_top  = trace_agg.head(50)
log_agg_top    = log_agg.head(50)

# error_logs summary
error_total_rows = len(error_df)
error_in_window = int(error_df['timestamp'].between(window_start, window_end).sum())
top_cmdb = error_df['cmdb_id'].value_counts().head(10).rename_axis('cmdb_id').reset_index(name='count')

# compose a compact summary DataFrame for error logs
error_summary_df = pd.DataFrame({
    'total_rows': [error_total_rows],
    'rows_in_window': [error_in_window]
})
# attach top cmdb as separate small table (keeps outputs compact)
top_cmdb_df = top_cmdb

# display results (variables returned)
metric_agg_top, trace_agg_top, log_agg_top, error_summary_df, top_cmdb_df

```
Out[1]:
```
Summary of results (straightforward):

- I produced aggregated tables for metric, trace and log (top 50 rows each) and an error_logs summary.
- Error logs: 13 total rows, all 13 occurred inside the incident window, and all 13 error_log entries are from the frontend service.
- Logs: frontend.log.error_count shows non-zero errors during the window (total_points=25, points_in_window=25, global_p95=4.0). Most other services have log.error_count = 0.0.
- Metrics/traces: many KPI/trace series were returned (top 50 each). One metric of note in the results is productcatalogservice.socket (global_p95 = 4.0), but the clearest signal of user-visible failure is the frontend error logs and log.error_count.

Conclusion / next step:
- Investigate the frontend service first (review its error log messages and related traces). Also check the productcatalogservice socket metric as a secondary lead.

The original code execution output of IPython Kernel is also provided below for reference:

(    source                cmdb_id        name  total_points    global_p95  points_in_window               earliest_ts                 latest_ts
0   metric              adservice         cpu            25  3.770899e+00                25 2024-02-21 04:07:00+00:00 2024-02-21 04:31:00+00:00
1   metric              adservice      diskio            25  7.496509e+04                25 2024-02-21 04:07:00+00:00 2024-02-21 04:31:00+00:00
2   metric              adservice  latency-50            25  3.969981e-03                25 2024-02-21 04:07:00+00:00 2024-02-21 04:31:00+00:00
3   metric              adservice  latency-90            25  8.657957e-03                25 2024-02-21 04:07:00+00:00 2024-02-21 04:31:00+00:00
4   metric              adservice         mem            25  1.026099e+08                25 2024-02-21 04:07:00+00:00 2024-02-21 04:31:00+00:00
..     ...                    ...         ...           ...           ...               ...                       ...                       ...
45  metric  productcatalogservice         cpu            25  3.847269e+00                25 2024-02-21 04:07:00+00:00 2024-02-21 04:31:00+00:00
46  metric  productcatalogservice  latency-50            25  1.799472e-03                25 2024-02-21 04:07:00+00:00 2024-02-21 04:31:00+00:00
47  metric  productcatalogservice  latency-90            25  4.395736e-03                25 2024-02-21 04:07:00+00:00 2024-02-21 04:31:00+00:00
48  metric  productcatalogservice         mem            25  1.031459e+07                25 2024-02-21 04:07:00+00:00 2024-02-21 04:31:00+00:00
49  metric  productcatalogservice      socket            25  4.000000e+00                25 2024-02-21 04:07:00+00:00 2024-02-21 04:31:00+00:00

[50 rows x 8 columns],    source                cmdb_id                                            name  total_points   global_p95  points_in_window               earliest_ts                 latest_ts
0   trace        checkoutservice        trace.from_checkoutservice.duration_mean            25     0.227712                25 2024-02-21 04:07:00+00:00 2024-02-21 04:31:00+00:00
1   trace        checkoutservice         trace.from_checkoutservice.duration_p95            25     1.193941                25 2024-02-21 04:07:00+00:00 2024-02-21 04:31:00+00:00
2   trace        checkoutservice           trace.from_checkoutservice.error_rate            25     0.000000                25 2024-02-21 04:07:00+00:00 2024-02-21 04:31:00+00:00
3   trace        checkoutservice            trace.from_checkoutservice.row_count            25   306.400000                25 2024-02-21 04:07:00+00:00 2024-02-21 04:31:00+00:00
4   trace        checkoutservice          trace.to_checkoutservice.duration_mean            25     0.227712                25 2024-02-21 04:07:00+00:00 2024-02-21 04:31:00+00:00
..    ...                    ...                                             ...           ...          ...               ...                       ...                       ...
45  trace  productcatalogservice         trace.from_frontendservice.duration_p95            25     0.000025                25 2024-02-21 04:07:00+00:00 2024-02-21 04:31:00+00:00
46  trace  productcatalogservice           trace.from_frontendservice.error_rate            25     0.000000                25 2024-02-21 04:07:00+00:00 2024-02-21 04:31:00+00:00
47  trace  productcatalogservice            trace.from_frontendservice.row_count            25  3863.800000                25 2024-02-21 04:07:00+00:00 2024-02-21 04:31:00+00:00
48  trace  productcatalogservice  trace.from_recommendationservice.duration_mean            25     0.000019                25 2024-02-21 04:07:00+00:00 2024-02-21 04:31:00+00:00
49  trace  productcatalogservice   trace.from_recommendationservice.duration_p95            25     0.000031                25 2024-02-21 04:07:00+00:00 2024-02-21 04:31:00+00:00

[50 rows x 8 columns],    source                cmdb_id             name  total_points  global_p95  points_in_window               earliest_ts                 latest_ts
0     log              adservice  log.error_count            25         0.0                25 2024-02-21 04:07:00+00:00 2024-02-21 04:31:00+00:00
1     log              adservice    log.row_count            25       469.8                25 2024-02-21 04:07:00+00:00 2024-02-21 04:31:00+00:00
2     log            cartservice  log.error_count            25         0.0                25 2024-02-21 04:07:00+00:00 2024-02-21 04:31:00+00:00
3     log            cartservice    log.row_count            25       815.8                25 2024-02-21 04:07:00+00:00 2024-02-21 04:31:00+00:00
4     log        checkoutservice  log.error_count            25         0.0                25 2024-02-21 04:07:00+00:00 2024-02-21 04:31:00+00:00
5     log        checkoutservice    log.row_count            25       106.8                25 2024-02-21 04:07:00+00:00 2024-02-21 04:31:00+00:00
6     log        currencyservice  log.error_count            25         0.0                25 2024-02-21 04:07:00+00:00 2024-02-21 04:31:00+00:00
7     log        currencyservice    log.row_count            25      2533.8                25 2024-02-21 04:07:00+00:00 2024-02-21 04:31:00+00:00
8     log           emailservice  log.error_count            25         0.0                25 2024-02-21 04:07:00+00:00 2024-02-21 04:31:00+00:00
9     log           emailservice    log.row_count            25        35.8                25 2024-02-21 04:07:00+00:00 2024-02-21 04:31:00+00:00
10    log               frontend  log.error_count            25         4.0                25 2024-02-21 04:07:00+00:00 2024-02-21 04:31:00+00:00
11    log               frontend    log.row_count            25      2785.8                25 2024-02-21 04:07:00+00:00 2024-02-21 04:31:00+00:00
12    log  recommendationservice  log.error_count            25         0.0                25 2024-02-21 04:07:00+00:00 2024-02-21 04:31:00+00:00
13    log  recommendationservice    log.row_count            25       593.6                25 2024-02-21 04:07:00+00:00 2024-02-21 04:31:00+00:00
14    log        shippingservice  log.error_count            25         0.0                25 2024-02-21 04:07:00+00:00 2024-02-21 04:31:00+00:00
15    log        shippingservice    log.row_count            25       513.6                25 2024-02-21 04:07:00+00:00 2024-02-21 04:31:00+00:00
16    log         paymentservice  log.error_count            24         0.0                24 2024-02-21 04:07:00+00:00 2024-02-21 04:30:00+00:00
17    log         paymentservice    log.row_count            24        72.0                24 2024-02-21 04:07:00+00:00 2024-02-21 04:30:00+00:00
18    log                  redis  log.error_count             5         0.0                 5 2024-02-21 04:10:00+00:00 2024-02-21 04:30:00+00:00
19    log                  redis    log.row_count             5         5.0                 5 2024-02-21 04:10:00+00:00 2024-02-21 04:30:00+00:00,    total_rows  rows_in_window
0          13              13,     cmdb_id  count
0  frontend     13)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[2]:

# reuse existing DataFrames: metric_df, trace_df, log_df, error_df
import pandas as pd
candidates = ['checkoutservice','currencyservice','emailservice','productcatalogservice','recommendationservice']

# incident window (reuse if defined, otherwise set)
try:
    window_start, window_end
except NameError:
    window_start = pd.Timestamp("2024-02-21 04:04:01", tz="UTC")
    window_end   = pd.Timestamp("2024-02-21 04:34:01", tz="UTC")

def aggregate_for_source(df, source, name_col, treat_error_count_special=False):
    # ensure value numeric
    df['value'] = pd.to_numeric(df['value'], errors='coerce')
    # thresholds (global p95) computed on entire series per group (rule 9)
    thresholds = df.groupby(['cmdb_id', name_col], dropna=False)['value'].quantile(0.95).reset_index().rename(columns={'value':'global_p95'})
    # restrict to candidate services
    thresholds = thresholds[thresholds['cmdb_id'].isin(candidates)].copy()
    if thresholds.empty:
        return pd.DataFrame(columns=['source','cmdb_id','name','global_p95','points_in_window','anomalies_in_window','earliest_anomaly_ts','max_value_in_window'])
    # windowed rows for candidates
    df_window = df[df['timestamp'].between(window_start, window_end) & df['cmdb_id'].isin(candidates)].copy()
    # merge thresholds to window rows for anomaly evaluation
    if not df_window.empty:
        df_window = df_window.merge(thresholds, on=['cmdb_id', name_col], how='right', suffixes=('','_thr'))
    else:
        # create empty merged with thresholds to allow consistent grouping (no in-window rows)
        df_window = thresholds.merge(pd.DataFrame(columns=df.columns), on=['cmdb_id', name_col], how='left', indicator=True)
        # ensure timestamp and value columns exist
        if 'timestamp' not in df_window.columns:
            df_window['timestamp'] = pd.NaT
        if 'value' not in df_window.columns:
            df_window['value'] = pd.NA
    # define anomaly condition per row
    # For metric/trace: anomaly = value >= global_p95
    # For logs: if treat_error_count_special True and name == 'log.error_count' -> anomaly value > 0 else >= global_p95
    def is_anomaly(row):
        v = row.get('value')
        gp = row.get('global_p95')
        name = row.get(name_col)
        if pd.isna(v) or pd.isna(gp):
            return False
        try:
            if source == 'log' and treat_error_count_special and name == 'log.error_count':
                return v > 0
            else:
                return v >= gp
        except Exception:
            return False
    # apply anomaly mask only on actual rows (rows coming from merge may be NaN rows)
    df_window['is_real_row'] = ~df_window['timestamp'].isna()
    df_window['anomaly'] = df_window.apply(lambda r: is_anomaly(r) if r['is_real_row'] else False, axis=1)
    # points_in_window per group (count of real rows in-window)
    points = df_window[df_window['is_real_row']].groupby(['cmdb_id', name_col]).size().rename('points_in_window')
    # anomalies_in_window
    anomalies = df_window[df_window['anomaly']].groupby(['cmdb_id', name_col]).size().rename('anomalies_in_window')
    # earliest anomaly timestamp
    earliest_anom = df_window[df_window['anomaly']].groupby(['cmdb_id', name_col])['timestamp'].min().rename('earliest_anomaly_ts')
    # max value in window (consider only real rows)
    max_in_window = df_window[df_window['is_real_row']].groupby(['cmdb_id', name_col])['value'].max().rename('max_value_in_window')
    # combine using thresholds index to include groups with zero in-window points
    out = thresholds.set_index(['cmdb_id', name_col]).join(points).join(anomalies).join(earliest_anom).join(max_in_window)
    out = out.reset_index().rename(columns={name_col:'name'})
    out['source'] = source
    # fill NaNs
    out['points_in_window'] = out['points_in_window'].fillna(0).astype(int)
    out['anomalies_in_window'] = out['anomalies_in_window'].fillna(0).astype(int)
    out['earliest_anomaly_ts'] = pd.to_datetime(out['earliest_anomaly_ts'])
    # reorder columns
    out = out[['source','cmdb_id','name','global_p95','points_in_window','anomalies_in_window','earliest_anomaly_ts','max_value_in_window']]
    return out

# aggregate each source for candidate services
metric_candidates_agg = aggregate_for_source(metric_df, 'metric', 'kpi_name', treat_error_count_special=False)
trace_candidates_agg  = aggregate_for_source(trace_df,  'trace',  'trace_name', treat_error_count_special=False)
log_candidates_agg    = aggregate_for_source(log_df,    'log',    'log_name', treat_error_count_special=True)

# combine and sort by anomalies desc then max_value_in_window desc
combined = pd.concat([metric_candidates_agg, trace_candidates_agg, log_candidates_agg], ignore_index=True, sort=False)
# ensure numeric for sorting
combined['max_value_in_window'] = pd.to_numeric(combined['max_value_in_window'], errors='coerce').fillna(-float('inf'))
combined_sorted = combined.sort_values(['anomalies_in_window','max_value_in_window'], ascending=[False, False]).reset_index(drop=True)

# limit to top 20 rows for compactness
combined_top20 = combined_sorted.head(20)

# recompute error_logs summary for reference
error_total_rows = len(error_df)
error_in_window = int(error_df['timestamp'].between(window_start, window_end).sum())
top_cmdb = error_df['cmdb_id'].value_counts().head(10).rename_axis('cmdb_id').reset_index(name='count')
error_summary_df = pd.DataFrame({'total_rows':[error_total_rows],'rows_in_window':[error_in_window]})
top_cmdb_df = top_cmdb

# return compact results
combined_top20, error_summary_df, top_cmdb_df

```
Out[2]:
```
Summary of findings for the five candidate services (straightforward):

Key signals
- Strongest signal: productcatalogservice.socket — all 25 in-window points meet/exceed the global p95 (anomalies_in_window = 25, max = 4.0). This is the top anomaly.
- Other socket anomalies: currencyservice.socket (25 anomalies, max 3.0) and emailservice.socket (25 anomalies, max 3.0).
- Disk I/O: emailservice.diskio and recommendationservice.diskio show anomalies (global_p95 = 0 and all in-window points flagged).
- Trace row_count spikes: productcatalogservice.trace.from_root.row_count (anomalies 24, max 30), emailservice.trace.from_root.row_count (24 anomalies, max 26), recommendationservice.trace.from_root.row_count (24 anomalies, max 24), currencyservice.trace.from_root.row_count (24 anomalies, max 20).
- Many trace error_rate series also flagged as anomalies because their global_p95 = 0, so every non-null in-window point meets the >= threshold; these zero-threshold error_rate flags should be treated cautiously (possible artifact of p95=0).

Error log summary (reference)
- error_logs: 13 total rows, all 13 within the incident window; all 13 error_log entries are from the frontend service.

Interpretation and recommended next steps (concise)
1. Investigate productcatalogservice first — the socket metric is the clearest abnormal signal (highest anomaly count and max value). Check productcatalogservice logs for socket/socket-connection errors and resource saturation.
2. Check network/socket issues between frontend and productcatalogservice (and between frontend and currency/email services) because multiple services show elevated socket metrics.
3. Verify productcatalogservice trace traffic: trace.from_root.row_count and max values indicate elevated request volume or spikes — confirm whether the service is overloaded.
4. Treat trace error_rate flags with caution when global_p95 == 0; instead inspect raw trace error events / logs for real errors.
5. Since frontend has all error_log entries in the window, correlate frontend errors with frontend → productcatalogservice (and frontend → currency/email) calls using traces to confirm the causal path.
6. If socket errors are confirmed, examine network, connection limits, and service thread/connection pools on productcatalogservice and the upstream frontend.

Bottom line: prioritize investigating productcatalogservice socket errors (then currencyservice/emailservice sockets), correlate with frontend error logs and traces to confirm causality.

The original code execution output of IPython Kernel is also provided below for reference:

(    source                cmdb_id                                         name  global_p95  points_in_window  anomalies_in_window       earliest_anomaly_ts  max_value_in_window
0   metric  productcatalogservice                                       socket         4.0                25                   25 2024-02-21 04:07:00+00:00                  4.0
1   metric        currencyservice                                       socket         3.0                25                   25 2024-02-21 04:07:00+00:00                  3.0
2   metric           emailservice                                       socket         3.0                25                   25 2024-02-21 04:07:00+00:00                  3.0
3   metric           emailservice                                       diskio         0.0                25                   25 2024-02-21 04:07:00+00:00                  0.0
4   metric  recommendationservice                                       diskio         0.0                25                   25 2024-02-21 04:07:00+00:00                  0.0
5    trace        checkoutservice        trace.from_checkoutservice.error_rate         0.0                25                   25 2024-02-21 04:07:00+00:00                  0.0
6    trace        checkoutservice          trace.to_checkoutservice.error_rate         0.0                25                   25 2024-02-21 04:07:00+00:00                  0.0
7    trace        checkoutservice             trace.to_emailservice.error_rate         0.0                25                   25 2024-02-21 04:07:00+00:00                  0.0
8    trace        currencyservice        trace.from_frontendservice.error_rate         0.0                25                   25 2024-02-21 04:07:00+00:00                  0.0
9    trace           emailservice        trace.from_checkoutservice.error_rate         0.0                25                   25 2024-02-21 04:07:00+00:00                  0.0
10   trace  productcatalogservice        trace.from_frontendservice.error_rate         0.0                25                   25 2024-02-21 04:07:00+00:00                  0.0
11   trace  productcatalogservice  trace.from_recommendationservice.error_rate         0.0                25                   25 2024-02-21 04:07:00+00:00                  0.0
12   trace  recommendationservice        trace.from_frontendservice.error_rate         0.0                25                   25 2024-02-21 04:07:00+00:00                  0.0
13   trace  recommendationservice  trace.from_recommendationservice.error_rate         0.0                25                   25 2024-02-21 04:07:00+00:00                  0.0
14   trace  recommendationservice    trace.to_productcatalogservice.error_rate         0.0                25                   25 2024-02-21 04:07:00+00:00                  0.0
15   trace  recommendationservice    trace.to_recommendationservice.error_rate         0.0                25                   25 2024-02-21 04:07:00+00:00                  0.0
16   trace  productcatalogservice                    trace.from_root.row_count        24.0                24                   24 2024-02-21 04:07:00+00:00                 30.0
17   trace           emailservice                    trace.from_root.row_count        24.0                24                   24 2024-02-21 04:07:00+00:00                 26.0
18   trace  recommendationservice                    trace.from_root.row_count        24.0                24                   24 2024-02-21 04:07:00+00:00                 24.0
19   trace        currencyservice                    trace.from_root.row_count        12.0                24                   24 2024-02-21 04:07:00+00:00                 20.0,    total_rows  rows_in_window
0          13              13,     cmdb_id  count
0  frontend     13)```
```

In [ ]:
# In[3]:

import pandas as pd

# ensure metric_df is loaded and timestamps parsed (reuse if present)
try:
    metric_df
except NameError:
    metric_df = pd.read_csv("metric.csv")
    metric_df['timestamp'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True)

# ensure timestamp dtype is correct
if metric_df['timestamp'].dtype == 'int64' or metric_df['timestamp'].dtype == 'float64':
    metric_df['timestamp'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True)

# incident window (UTC)
window_start = pd.Timestamp("2024-02-21 04:04:01", tz="UTC")
window_end   = pd.Timestamp("2024-02-21 04:34:01", tz="UTC")

# filter full series for productcatalogservice.socket (do NOT filter by window for p95)
mask_all = (metric_df['cmdb_id'] == 'productcatalogservice') & metric_df['kpi_name'].str.contains('socket', na=False)
series_all = metric_df[mask_all].copy()

# compute global p95 over the full series (rule 9)
if not series_all.empty:
    series_all['value'] = pd.to_numeric(series_all['value'], errors='coerce')
    global_p95 = float(series_all['value'].quantile(0.95))
else:
    global_p95 = float('nan')

# filter rows within the incident window for productcatalogservice socket KPIs
mask_window = mask_all & metric_df['timestamp'].between(window_start, window_end)
in_window = metric_df.loc[mask_window, ['timestamp','cmdb_id','kpi_name','value']].copy()
in_window['value'] = pd.to_numeric(in_window['value'], errors='coerce')
in_window = in_window.sort_values('timestamp').reset_index(drop=True)

# compute requested stats
points_in_window = int(len(in_window))
if points_in_window > 0 and not pd.isna(global_p95):
    anomaly_mask = in_window['value'] >= global_p95
    anomalies_in_window = int(anomaly_mask.sum())
    if anomalies_in_window > 0:
        earliest_anomaly_ts = in_window.loc[anomaly_mask, 'timestamp'].min()
        latest_anomaly_ts   = in_window.loc[anomaly_mask, 'timestamp'].max()
    else:
        earliest_anomaly_ts = pd.NaT
        latest_anomaly_ts = pd.NaT
    max_value_in_window = float(in_window['value'].max())
    # compute longest consecutive anomaly run in minutes
    # consider only anomaly rows and compute runs where consecutive timestamps differ by exactly 60 seconds
    if anomalies_in_window == 0:
        longest_run = 0
    else:
        anom_ts = in_window.loc[anomaly_mask, 'timestamp'].sort_values().reset_index(drop=True)
        # compute diffs in seconds
        diffs = anom_ts.diff().dt.total_seconds().fillna(9999)
        # start a new group where diff != 60
        group_id = (diffs != 60).cumsum()
        run_lengths = anom_ts.groupby(group_id).size()
        longest_run = int(run_lengths.max())
else:
    anomalies_in_window = 0
    earliest_anomaly_ts = pd.NaT
    latest_anomaly_ts = pd.NaT
    max_value_in_window = float('nan')
    longest_run = 0

# prepare compact summary DataFrame (single-row)
summary_df = pd.DataFrame([{
    'cmdb_id': 'productcatalogservice',
    'kpi': 'socket (kpi_name contains "socket")',
    'global_p95': global_p95,
    'points_in_window': points_in_window,
    'anomalies_in_window': anomalies_in_window,
    'earliest_anomaly_ts': earliest_anomaly_ts,
    'latest_anomaly_ts': latest_anomaly_ts,
    'longest_consecutive_anomaly_run_in_minutes': longest_run,
    'max_value_in_window': max_value_in_window
}])

# prepare in-window rows limited to top 50 (show timestamp and value)
in_window_rows = in_window[['timestamp','kpi_name','value']].head(50)

# return compact results
summary_df, in_window_rows

```
Out[3]:
```
Summary for productcatalogservice.socket (UTC):

- global p95 (full series, not window-filtered): 4.0
- points in incident window (2024-02-21 04:04:01 → 2024-02-21 04:34:01 UTC): 25
- anomalies in window (value >= global_p95): 25 (every in-window point)
- earliest anomaly timestamp: 2024-02-21 04:07:00+00:00
- latest anomaly timestamp: 2024-02-21 04:31:00+00:00
- longest consecutive anomaly run: 25 minutes (continuous, adjacent-minute anomalies)
- max value in window: 4.0
- In-window rows (timestamp, value): one row per minute from 2024-02-21 04:07:00+00:00 through 2024-02-21 04:31:00+00:00, each value = 4.0 (25 rows).

Interpretation: The socket metric for productcatalogservice is consistently at or above its global p95 for the entire incident window — a sustained, minute-by-minute anomaly (value = 4) from 04:07 to 04:31 UTC. This is a strong, persistent signal to investigate socket/network/connection issues on productcatalogservice.

The original code execution output of IPython Kernel is also provided below for reference:

(                 cmdb_id                                  kpi  global_p95  points_in_window  anomalies_in_window       earliest_anomaly_ts         latest_anomaly_ts  longest_consecutive_anomaly_run_in_minutes  max_value_in_window
0  productcatalogservice  socket (kpi_name contains "socket")         4.0                25                   25 2024-02-21 04:07:00+00:00 2024-02-21 04:31:00+00:00                                          25                  4.0,                    timestamp kpi_name  value
0  2024-02-21 04:07:00+00:00   socket    4.0
1  2024-02-21 04:08:00+00:00   socket    4.0
2  2024-02-21 04:09:00+00:00   socket    4.0
3  2024-02-21 04:10:00+00:00   socket    4.0
4  2024-02-21 04:11:00+00:00   socket    4.0
..                       ...      ...    ...
20 2024-02-21 04:27:00+00:00   socket    4.0
21 2024-02-21 04:28:00+00:00   socket    4.0
22 2024-02-21 04:29:00+00:00   socket    4.0
23 2024-02-21 04:30:00+00:00   socket    4.0
24 2024-02-21 04:31:00+00:00   socket    4.0

[25 rows x 3 columns])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[4]:

import pandas as pd

# reuse existing trace_df and error_df if present; otherwise load
try:
    trace_df
except NameError:
    trace_df = pd.read_csv("trace.csv")
    trace_df['timestamp'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True)

try:
    error_df
except NameError:
    error_df = pd.read_csv("error_logs.csv")
    error_df['timestamp'] = pd.to_datetime(error_df['timestamp'], unit='s', utc=True)

# ensure timestamps parsed and value numeric
if trace_df['timestamp'].dtype != 'datetime64[ns, UTC]':
    trace_df['timestamp'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True)
trace_df['value'] = pd.to_numeric(trace_df['value'], errors='coerce')

if error_df['timestamp'].dtype != 'datetime64[ns, UTC]':
    error_df['timestamp'] = pd.to_datetime(error_df['timestamp'], unit='s', utc=True)

# incident window (UTC)
window_start = pd.Timestamp("2024-02-21 04:04:01", tz="UTC")
window_end   = pd.Timestamp("2024-02-21 04:34:01", tz="UTC")

# selection: frontend -> productcatalogservice traces
sel_mask = (
    ((trace_df['cmdb_id'] == 'productcatalogservice') & trace_df['trace_name'].str.contains('from_frontendservice', na=False)) |
    ((trace_df['cmdb_id'] == 'frontend') & trace_df['trace_name'].str.contains('to_productcatalogservice', na=False))
)

selected_pairs = trace_df.loc[sel_mask, ['cmdb_id', 'trace_name']].drop_duplicates()

# compute global p95 for all series, then filter to selected pairs (rule: compute p95 on full series)
p95_all = trace_df.groupby(['cmdb_id', 'trace_name'], dropna=False)['value'].quantile(0.95).reset_index().rename(columns={'value':'global_p95'})
p95_selected = p95_all.merge(selected_pairs, on=['cmdb_id','trace_name'], how='inner')

# filter in-window rows for selected series
window_mask = trace_df['timestamp'].between(window_start, window_end) & sel_mask
in_window = trace_df.loc[window_mask, ['timestamp','cmdb_id','trace_name','value']].copy()

# merge global_p95 into in-window rows to evaluate anomalies
if not in_window.empty:
    in_window = in_window.merge(p95_selected, on=['cmdb_id','trace_name'], how='left')
    in_window['is_anomaly'] = in_window['value'] >= in_window['global_p95']
else:
    # create empty consistent DataFrame to allow aggregations (will be joined with p95_selected later)
    in_window = pd.DataFrame(columns=['timestamp','cmdb_id','trace_name','value','global_p95','is_anomaly'])

# aggregate in-window stats per series
points = in_window.groupby(['cmdb_id','trace_name']).size().rename('points_in_window')
anomalies = in_window[in_window['is_anomaly']].groupby(['cmdb_id','trace_name']).size().rename('anomalies_in_window')
earliest_anom = in_window[in_window['is_anomaly']].groupby(['cmdb_id','trace_name'])['timestamp'].min().rename('earliest_anomaly_ts')
max_val = in_window.groupby(['cmdb_id','trace_name'])['value'].max().rename('max_value_in_window')

# combine with p95_selected to include series that may have zero in-window points
agg = p95_selected.set_index(['cmdb_id','trace_name']).join(points).join(anomalies).join(earliest_anom).join(max_val).reset_index()
agg['points_in_window'] = agg['points_in_window'].fillna(0).astype(int)
agg['anomalies_in_window'] = agg['anomalies_in_window'].fillna(0).astype(int)
agg['earliest_anomaly_ts'] = pd.to_datetime(agg['earliest_anomaly_ts'])
agg['max_value_in_window'] = pd.to_numeric(agg['max_value_in_window'], errors='coerce')

# final columns and source label
agg = agg.rename(columns={'trace_name':'name'})
agg.insert(0, 'source', 'trace')
agg = agg[['source','cmdb_id','name','global_p95','points_in_window','anomalies_in_window','earliest_anomaly_ts','max_value_in_window']]

# sort by anomalies desc then max_value desc and limit to top 20
trace_agg_top = agg.sort_values(['anomalies_in_window','max_value_in_window'], ascending=[False, False]).reset_index(drop=True).head(20)

# extract frontend error_logs within the incident window (timestamp and message), limit to 50 rows
frontend_error_mask = (error_df['cmdb_id'] == 'frontend') & error_df['timestamp'].between(window_start, window_end)
frontend_errors_in_window = error_df.loc[frontend_error_mask, ['timestamp','message']].sort_values('timestamp').head(50).reset_index(drop=True)

# return results (compact)
trace_agg_top, frontend_errors_in_window

```
Out[4]:
```
Summary (straightforward):

Trace-series summary for frontend ↔ productcatalogservice (incident window 2024-02-21 04:04:01 → 04:34:01 UTC)
- trace.from_frontendservice.error_rate (cmdb_id=productcatalogservice)
  - global_p95 = 0.0 (computed on full series)
  - points_in_window = 25, anomalies_in_window flagged = 25 (artifact because global_p95 == 0)
  - max_value_in_window = 0.0
  - Interpretation: p95=0 makes the "anomaly" flag meaningless here — error_rate stayed 0 across the window.

- trace.from_frontendservice.duration_p95 (productcatalogservice)
  - global_p95 = 0.000025
  - points_in_window = 25, anomalies_in_window = 4
  - earliest anomaly at 2024-02-21 04:15:00+00:00
  - max_value_in_window = 0.000032
  - Interpretation: a few p95 latency spikes around ~04:15 UTC.

- trace.from_frontendservice.row_count (productcatalogservice)
  - global_p95 = 3863.8
  - points_in_window = 25, anomalies_in_window = 2
  - earliest anomaly at 2024-02-21 04:13:00+00:00
  - max_value_in_window = 3895.0
  - Interpretation: two high-traffic minutes (~04:13) but small in count.

- trace.from_frontendservice.duration_mean (productcatalogservice)
  - global_p95 = 0.000015
  - points_in_window = 25, anomalies_in_window = 2
  - earliest anomaly at 2024-02-21 04:30:00+00:00
  - max_value_in_window = 0.000017
  - Interpretation: a couple small mean-duration spikes near 04:30.

Frontend error logs within the incident window
- Extracted frontend error log rows (all within the window): 13 messages, timestamped between 2024-02-21 04:25:48+00:00 and 2024-02-21 04:26:05+00:00, message = "request error" (clustered around 04:25–04:26 UTC).

Overall interpretation and next steps (concise)
- productcatalogservice shows sustained socket anomalies (from prior metric analysis) and a few trace spikes (traffic and latency) earlier in the window (~04:13–04:15 and a small latency spike ~04:30).
- The frontend produced a concentrated burst of "request error" log messages at ~04:25–04:26 UTC.
- Correlation: trace spikes do not perfectly align with the frontend error burst; still, productcatalogservice is a strong candidate (socket anomalies + trace traffic/latency spikes). Also note the error_rate trace p95 = 0 (so its anomaly count is an artifact).
- Recommended actions: investigate productcatalogservice socket/connectivity and logs across 04:07–04:31 UTC, and inspect frontend error details at 04:25–04:26 to identify the failing calls and confirm causality.

The original code execution output of IPython Kernel is also provided below for reference:

(  source                cmdb_id                                      name   global_p95  points_in_window  anomalies_in_window       earliest_anomaly_ts  max_value_in_window
0  trace  productcatalogservice     trace.from_frontendservice.error_rate     0.000000                25                   25 2024-02-21 04:07:00+00:00             0.000000
1  trace  productcatalogservice   trace.from_frontendservice.duration_p95     0.000025                25                    4 2024-02-21 04:15:00+00:00             0.000032
2  trace  productcatalogservice      trace.from_frontendservice.row_count  3863.800000                25                    2 2024-02-21 04:13:00+00:00          3895.000000
3  trace  productcatalogservice  trace.from_frontendservice.duration_mean     0.000015                25                    2 2024-02-21 04:30:00+00:00             0.000017,                    timestamp        message
0  2024-02-21 04:25:48+00:00  request error
1  2024-02-21 04:25:50+00:00  request error
2  2024-02-21 04:25:53+00:00  request error
3  2024-02-21 04:25:55+00:00  request error
4  2024-02-21 04:25:55+00:00  request error
5  2024-02-21 04:25:55+00:00  request error
6  2024-02-21 04:25:59+00:00  request error
7  2024-02-21 04:25:59+00:00  request error
8  2024-02-21 04:26:00+00:00  request error
9  2024-02-21 04:26:01+00:00  request error
10 2024-02-21 04:26:02+00:00  request error
11 2024-02-21 04:26:03+00:00  request error
12 2024-02-21 04:26:05+00:00  request error)```
```